In [ ]:
import os
import duckdb
from dotenv import load_dotenv

load_dotenv()
con = duckdb.connect()

## 1. Install and attach Postgres to DuckDB

In [ ]:
con.sql("INSTALL postgres")
con.sql("LOAD postgres")

con.sql(f"""
    ATTACH 'host=192.168.0.204 port=5432 dbname=land_registry
            user={os.environ["PGUSER"]} password={os.environ["PGPASSWORD"]}'
    AS pg (TYPE postgres, READ_ONLY)
""")

con.sql("SHOW ALL TABLES").show()
con.sql("SELECT * FROM pg.public.staging_price_paid_cleaned LIMIT 5").show()

## 2. Count rows per district

In [ ]:
con.sql("""
    SELECT * FROM postgres_query('pg', '
        SELECT district_clean, count(*) AS n
        FROM staging_price_paid_cleaned
        GROUP BY 1
        ORDER BY 2 DESC
    ')
""").show(max_rows=40)

## 3. Save Bristol and Powys slices
Bristol is the dense urban slice; Powys is the rural one (many addresses with a house name and no street).

In [ ]:
os.makedirs("data", exist_ok=True)

slices = {
    "bristol": "CITY OF BRISTOL",
    "powys": "POWYS",
}

for name, district in slices.items():
    con.sql(f"""
        COPY (
            SELECT *
            FROM pg.public.staging_price_paid_cleaned
            WHERE district_clean = '{district}'
        ) TO 'data/{name}.parquet' (FORMAT parquet)
    """)

## 4. Reload from file on fresh DuckDB instance

In [ ]:
# Drop postgres attachment and read snapshot
con = duckdb.connect()
con.sql("CREATE VIEW slice AS SELECT * FROM 'data/bristol.parquet'")

con.sql("DESCRIBE slice").show(max_rows=30)
con.sql("SELECT count(*) AS n FROM slice").show()

## 5. NULL Rates per column

In [ ]:
cols = [r[0] for r in con.sql("DESCRIBE slice").fetchall()]

null_rates = " UNION ALL ".join(
    f"""SELECT '{c}' AS col,
               count(*) FILTER (WHERE {c} IS NULL) AS nulls,
               round(100.0 * count(*) FILTER (WHERE {c} IS NULL) / count(*), 2) AS pct
        FROM slice"""
    for c in cols
)
con.sql(f"SELECT * FROM ({null_rates}) ORDER BY pct DESC").show(max_rows=30)

# Check for empty strings
text_cols = [c for c in cols if c.endswith("_clean") or c == "flat_identifier"]
empties = " UNION ALL ".join(
    f"SELECT '{c}' AS col, count(*) FILTER (WHERE {c} = '') AS empty_strings FROM slice"
    for c in text_cols
)
con.sql(empties).show(max_rows=20)

## 6. How much does NULL SAON matter?
Flats are the problem (27.9% missing SAON)

In [ ]:
con.sql("""
    SELECT property_type,
           count(*) AS n,
           round(100.0 * count(*) FILTER (WHERE saon_clean IS NULL) / count(*), 1) AS pct_null_saon,
           round(100.0 * count(*) FILTER (WHERE flat_identifier IS NOT NULL) / count(*), 1) AS pct_flat_id
    FROM slice
    GROUP BY 1 ORDER BY 2 DESC
""").show()

## 7. Find flat-heavy buildings

In [ ]:
con.sql("""
    SELECT postcode_clean, paon_clean, street_clean,
           count(*) AS n_sales,
           count(DISTINCT saon_clean) AS n_saon
    FROM slice
    WHERE saon_clean IS NOT NULL
    GROUP BY ALL
    ORDER BY n_sales DESC
    LIMIT 10
""").show()

## 8. Pick row from above table and look at its transactions

In [ ]:
con.sql("""
    SELECT saon_clean, flat_identifier, property_type, date_of_transfer, price
    FROM slice
    WHERE postcode_clean = 'BS3 3NG' AND paon_clean = 'AIRPOINT'    -- change the postcode_clean and paon_clean fields
    ORDER BY saon_clean, date_of_transfer
""").show(max_rows=60)

In [ ]:
con.sql("""
    CREATE OR REPLACE VIEW linkage_input AS
    SELECT *, coalesce(flat_identifier, saon_clean, '<NONE>') AS unit_key
    FROM slice
    WHERE NOT (property_type IN ('F', 'O') AND saon_clean IS NULL)
""")

con.sql("""
    CREATE OR REPLACE VIEW unit_unknown AS
    SELECT * FROM slice
    WHERE property_type IN ('F', 'O') AND saon_clean IS NULL
""")

con.sql("""
    SELECT (SELECT count(*) FROM slice)         AS slice_rows,
           (SELECT count(*) FROM linkage_input) AS linkage_rows,
           (SELECT count(*) FROM unit_unknown)  AS unit_unknown_rows
""").show()

## 9. Export linkage input and unit-unknown for each slice
Same split as above, applied to every slice so Bristol (urban) and Powys (rural) go through identical rules.

In [ ]:
for name in ["bristol", "powys"]:
    con.sql(f"""
        COPY (
            SELECT *, coalesce(flat_identifier, saon_clean, '<NONE>') AS unit_key
            FROM 'data/{name}.parquet'
            WHERE NOT (property_type IN ('F', 'O') AND saon_clean IS NULL)
        ) TO 'data/{name}_linkage_input.parquet' (FORMAT parquet)
    """)
    con.sql(f"""
        COPY (
            SELECT * FROM 'data/{name}.parquet'
            WHERE property_type IN ('F', 'O') AND saon_clean IS NULL
        ) TO 'data/{name}_unit_unknown.parquet' (FORMAT parquet)
    """)

    con.sql(f"""
        SELECT '{name}' AS slice,
               (SELECT count(*) FROM 'data/{name}.parquet')               AS slice_rows,
               (SELECT count(*) FROM 'data/{name}_linkage_input.parquet') AS linkage_rows,
               (SELECT count(*) FROM 'data/{name}_unit_unknown.parquet')  AS unit_unknown_rows
    """).show()

## Key Findings

### 1. The same flat appears under two SAON spellings
In Airpoint (BS3 3NG, a large apartment block), 338 sales carry 309 distinct SAON strings but only 251 distinct `flat_identifier` values. 58 units (23% of the building) appear as both a bare number (`106`) and a prefixed form (`FLAT 106`). Without `flat_identifier`, each of those flats would receive two property IDs and its resales would never link.

*Open question:* the format may depend on the year of sale (in the rows I looked at, 2007-2009 sales used `FLAT 7xx` and later ones bare numbers). If so, the split falls between a unit's first sale and its resales, which are exactly the pairs the index needs. To be tested by looking at the share of `FLAT`-prefixed SAONs by year.

### 2. A same-day batch of discounted sales, all category A
25 Airpoint sales on 2009-04-17. Every row I inspected is category A, and nearly all are new builds. Five units sold for exactly £86,600 and several for exactly £118,000, against roughly £145k-£230k for neighbouring units in mid-2008. Probably a developer clearing unsold stock at a discount, recorded as ordinary sales (note the time period).

Filtering to category A will not remove this pattern, so the index may need to handle first sales of new builds separately. That is an econometrics decision for later, so `old_new` is carried through.

### 3. Category B is small
B is 12,431 of 244,848 rows (5.1%). That is too few to affect linkage, so B stays in the linkage input (property identity does not depend on transaction type, and dropping B would break ownership chains). Filtering by `ppd_category` happens downstream, so the flip detector can run with and without B.

### 4. Flats and `O` properties with no SAON are excluded from linkage
- 27.9% of flats (20,606 rows) have no SAON. At the address level, 347 addresses have six or more such sales (about 8 each), which almost certainly means several different flats sharing one address string.
- With NULL treated as "missing", a no-unit row gives no evidence against any neighbour. Clustering is transitive, so it could bridge two different flats into one property, creating a false repeat sale. A false merge is worse than a missed link, because it shows up as an outlier in the index and as a false positive in the flip detector.
- Decision: rows with `property_type IN ('F', 'O')` and no SAON go to a separate view (`unit_unknown`, 24,390 rows, 10.0% of the slice) and get no property ID. `O` is treated like flats because "no sub-unit" cannot safely be asserted for commercial or mixed-use property.
- All remaining rows get `unit_key = coalesce(flat_identifier, saon_clean, '<NONE>')`. For houses, `<NONE>` is a real fact (no sub-unit), so two houses at the same address agree on it.
- **Limitation:** these rows will be absent from the repeat-sales index and the flip detector. The share will vary by area.